# DETECT & TRACKING
Lấy video test tại: https://drive.google.com/drive/folders/194pi0dANgFZBJ5JwCOzOdWYg7sWG_31t?fbclid=IwY2xjawSmMpRleHRuA2FlbQIxMABicmlkETFHWTZxUmQ5UU9VSmJ5YUJTc3J0YwZhcHBfaWQQMjIyMDM5MTc4ODIwMDg5MgABHnL4u1t8Ukj5zwUe2YCauh0Hv5lc3hpDbYsP7QWATjp9rwA49PPly31rhMgf_aem_iEu0hYpDz-jFB3jeX3vQew

In [ ]:
!pip install -q ultralytics opencv-python lxml pyyaml tqdm

In [ ]:
from ultralytics import YOLO
import torch

# Kiểm tra MPS
if torch.backends.mps.is_available():
    device = "mps"
    print("Using Apple Silicon GPU (MPS)")
else:
    device = "cpu"
    print("Using CPU")

model = YOLO("yolo11n.pt")

tracking_results = model.track(
    source="20221003-115037.mp4",
    tracker="bytetrack.yaml",
    imgsz=640,
    conf=0.25,
    iou=0.5,
    device=device,
    save=True,
    save_txt=True,
    project="runs",
    name="yolo11n_bytetrack_ua_detrac",
    persist=True
)

print("Done")

Using Apple Silicon GPU (MPS)

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/736) /Users/vivianele/Desktop/CITD_thesis/20221003-115037.mp4: 384x640 5 persons, 4 cars, 6 motorcycles, 1 bus, 1 potted plant, 309.2ms
video 1/1 (frame 2/736) /Users/vivianele/Desktop/CITD_thesis/20221003-115037.mp4: 384x640 5 persons, 4 cars, 6 motorcycles, 1 bus, 1 potted plant, 20.1ms
video 1/1 (frame 3/736) /Users/vivianele/Desktop/CITD_thesis/20221003-115037.mp4: 384x640 5 persons, 4 cars, 5 motorcyc

# LẤY VỊ TRÍ CỦA CÁC XE QUA CÁC FRAMES

In [ ]:
import pandas as pd
from ultralytics import YOLO

video_path = "20221003-115037.mp4"
model = YOLO("yolo11n.pt")

# các class vehicle trong COCO
VEHICLE_CLASSES = {
    "car",
    "truck",
    "bus",
    "bicycle",
    "motorcycle"
}

rows = []

results = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    conf=0.25,
    persist=True,
    stream=True
)

frame_id = 0

for r in results:

    frame_id += 1

    if r.boxes is None:
        continue

    names = r.names

    boxes = r.boxes

    if boxes.id is None:
        continue

    track_ids = boxes.id.cpu().numpy().astype(int)
    classes = boxes.cls.cpu().numpy().astype(int)
    confs = boxes.conf.cpu().numpy()

    xyxy = boxes.xyxy.cpu().numpy()

    for track_id, cls_id, conf, box in zip(
        track_ids,
        classes,
        confs,
        xyxy
    ):

        class_name = names[int(cls_id)]

        # chỉ giữ phương tiện
        if class_name not in VEHICLE_CLASSES:
            continue

        x1, y1, x2, y2 = box

        rows.append({
            "frame": frame_id,
            "track_id": int(track_id),
            "class_name": class_name,

            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),

            "center_x": float((x1 + x2) / 2),
            "center_y": float((y1 + y2) / 2),
            "bottom_x": float((x1 + x2) / 2),
            "bottom_y": float(y2),
        })

df = pd.DataFrame(rows)

df.to_csv(
    "results/vehicle_positions_by_frame.csv",
    index=False
)

print(df.head())
print("Total detections:", len(df))


video 1/1 (frame 1/736) /Users/vivianele/Desktop/CITD_thesis/20221003-115037.mp4: 384x640 5 persons, 4 cars, 9 motorcycles, 1 bus, 1 potted plant, 44.2ms
video 1/1 (frame 2/736) /Users/vivianele/Desktop/CITD_thesis/20221003-115037.mp4: 384x640 5 persons, 4 cars, 8 motorcycles, 1 bus, 1 potted plant, 31.8ms
video 1/1 (frame 3/736) /Users/vivianele/Desktop/CITD_thesis/20221003-115037.mp4: 384x640 6 persons, 4 cars, 6 motorcycles, 1 bus, 1 potted plant, 30.1ms
video 1/1 (frame 4/736) /Users/vivianele/Desktop/CITD_thesis/20221003-115037.mp4: 384x640 6 persons, 4 cars, 5 motorcycles, 1 bus, 1 potted plant, 31.6ms
video 1/1 (frame 5/736) /Users/vivianele/Desktop/CITD_thesis/20221003-115037.mp4: 384x640 6 persons, 4 cars, 3 motorcycles, 1 bus, 1 potted plant, 31.1ms
video 1/1 (frame 6/736) /Users/vivianele/Desktop/CITD_thesis/20221003-115037.mp4: 384x640 5 persons, 4 cars, 3 motorcycles, 1 bus, 31.8ms
video 1/1 (frame 7/736) /Users/vivianele/Desktop/CITD_thesis/20221003-115037.mp4: 384x640 7

# VẼ LANE

In [ ]:
import cv2
import json
from pathlib import Path

video_path = "20221003-115037.mp4"   # sửa path video
lane_json = "results/lanes.json"

cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

if not ret:
    raise ValueError("Không đọc được video")

points = []
lanes = []
current_lane = []

def mouse_callback(event, x, y, flags, param):
    global current_lane, lanes

    if event == cv2.EVENT_LBUTTONDOWN:
        current_lane.append([x, y])
        print("Point:", x, y)

    elif event == cv2.EVENT_RBUTTONDOWN:
        if len(current_lane) >= 3:
            lane_name = f"lane_{len(lanes)+1}"
            lanes.append({
                "name": lane_name,
                "points": current_lane.copy()
            })
            print(f"Saved {lane_name}:", current_lane)
            current_lane = []
        else:
            print("Cần ít nhất 3 điểm để tạo 1 lane")

clone = frame.copy()
cv2.namedWindow("Draw lanes")
cv2.setMouseCallback("Draw lanes", mouse_callback)

print("""
Hướng dẫn:
- Click trái: thêm điểm polygon cho lane
- Click phải: kết thúc lane hiện tại
- Nhấn S: lưu lanes.json
- Nhấn Q: thoát
""")

while True:
    display = clone.copy()

    # vẽ lane đã lưu
    for lane in lanes:
        pts = lane["points"]
        for i in range(len(pts)):
            cv2.line(display, tuple(pts[i]), tuple(pts[(i+1) % len(pts)]), (0,255,0), 2)
        cv2.putText(display, lane["name"], tuple(pts[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    # vẽ lane đang chọn
    for p in current_lane:
        cv2.circle(display, tuple(p), 5, (0,0,255), -1)
    for i in range(len(current_lane)-1):
        cv2.line(display, tuple(current_lane[i]), tuple(current_lane[i+1]), (0,0,255), 2)

    cv2.imshow("Draw lanes", display)
    key = cv2.waitKey(1) & 0xFF

    if key == ord("s"):
        with open(lane_json, "w") as f:
            json.dump(lanes, f, indent=4)
        print("Saved:", lane_json)

    elif key == ord("q"):
        break

cv2.destroyAllWindows()


Hướng dẫn:
- Click trái: thêm điểm polygon cho lane
- Click phải: kết thúc lane hiện tại
- Nhấn S: lưu lanes.json
- Nhấn Q: thoát

Point: 552 346
Point: 430 349
Point: 428 48
Point: 445 47
Saved lane_1: [[552, 346], [430, 349], [428, 48], [445, 47]]
Point: 430 349
Point: 314 347
Point: 406 50
Point: 428 50
Saved lane_2: [[430, 349], [314, 347], [406, 50], [428, 50]]
Saved: data/lanes.json


# PHÁT HIỆN VI PHẠM

In [ ]:
import json
import cv2
import numpy as np
import pandas as pd

positions_csv = "results/vehicle_positions_by_frame.csv"
lane_json = "results/lanes.json"

output_detail_csv = "results/traffic_violations_detail.csv"
output_summary_csv = "results/traffic_violations_summary.csv"

# =========================
# 1) CẤU HÌNH RULE
# =========================

# Lỗi đi sai làn theo loại xe
LANE_RULES = {
    "lane_1": {"motorcycle", "bicycle"},
    "lane_2": {"car", "bus", "truck", "van"},
}

# Làn hợp lệ cho xe đi hướng bên phải: từ dưới lên trên
RIGHT_DIRECTION_LANES = {"lane_1", "lane_2"}

MIN_DELTA_Y = 20
MIN_VIOLATION_FRAMES = 10


# =========================
# 2) LOAD DATA
# =========================

df = pd.read_csv(positions_csv)

with open(lane_json, "r") as f:
    lanes = json.load(f)


# =========================
# 3) HÀM PHỤ
# =========================

def get_lane_name(x, y, lanes):
    for lane in lanes:
        pts = np.array(lane["points"], dtype=np.int32)
        inside = cv2.pointPolygonTest(pts, (float(x), float(y)), False)
        if inside >= 0:
            return lane["name"]
    return "unknown"


def estimate_direction(track_df, min_delta_y=20):
    track_df = track_df.sort_values("frame")

    y_start = track_df["bottom_y"].iloc[0]
    y_end = track_df["bottom_y"].iloc[-1]

    delta_y = y_end - y_start

    if delta_y < -min_delta_y:
        return "up"
    elif delta_y > min_delta_y:
        return "down"
    else:
        return "static_or_unclear"


def is_wrong_lane(class_name, lane_name, lane_rules):
    if lane_name == "unknown":
        return False

    allowed_classes = lane_rules.get(lane_name)

    if allowed_classes is None:
        return False

    return class_name not in allowed_classes


# =========================
# 4) XÁC ĐỊNH LANE
# =========================

df["lane_name"] = df.apply(
    lambda row: get_lane_name(row["bottom_x"], row["bottom_y"], lanes),
    axis=1
)


# =========================
# 5) XÁC ĐỊNH HƯỚNG DI CHUYỂN
# =========================

direction_rows = []

for track_id, g in df.groupby("track_id"):
    direction_rows.append({
        "track_id": track_id,
        "direction": estimate_direction(g, MIN_DELTA_Y)
    })

df_direction = pd.DataFrame(direction_rows)
df = df.merge(df_direction, on="track_id", how="left")


# =========================
# 6) PHÁT HIỆN 2 LOẠI LỖI
# =========================

df["wrong_lane"] = df.apply(
    lambda row: is_wrong_lane(
        row["class_name"],
        row["lane_name"],
        LANE_RULES
    ),
    axis=1
)

# Làn 1 và làn 2 là làn dành cho hướng đi lên
UP_DIRECTION_LANES = {"lane_1", "lane_2"}

# Nếu xe đi xuống mà nằm trong lane_1 hoặc lane_2 => đi ngược chiều
# Nếu xe đi lên mà nằm ngoài lane 2 => đi ngược chiều 
df["wrong_direction"] = (
    (df["direction"] == "down") &
    (df["lane_name"].isin(RIGHT_DIRECTION_LANES))
)| (
    (df["direction"] == "up") &
    (~df["lane_name"].isin(RIGHT_DIRECTION_LANES))
)


# =========================
# 7) GỘP LỖI THÀNH 1 CỘT violation_type
# =========================

def get_violation_type(row):

    # Ưu tiên lỗi nặng nhất
    if row["wrong_direction"]:
        return "wrong_direction"

    if row["wrong_lane"]:
        return "wrong_lane"

    return "none"

df["violation_type"] = df.apply(get_violation_type, axis=1)
df["has_violation"] = df["violation_type"] != "none"


# =========================
# 8) LƯU DETAIL THEO FRAME
# =========================

df_detail = df[df["has_violation"]].copy()

df_detail.to_csv(output_detail_csv, index=False)

print("Saved detail:", output_detail_csv)
print(df_detail.head())


# =========================
# 9) LƯU SUMMARY THEO TỪNG XE
# =========================

summary = (
    df_detail
    .groupby(["track_id", "class_name", "violation_type"])
    .agg(
        first_frame=("frame", "min"),
        last_frame=("frame", "max"),
        violation_frames=("frame", "count"),
        lanes_used=("lane_name", lambda x: ",".join(sorted(set(x)))),
        direction=("direction", "first")
    )
    .reset_index()
)

summary = summary[summary["violation_frames"] >= MIN_VIOLATION_FRAMES]

summary.to_csv(output_summary_csv, index=False)

print("Saved summary:", output_summary_csv)
summary

Saved detail: traffic_violations_detail.csv
    frame  track_id class_name          x1         y1          x2          y2  \
1       1         2        car  440.235596  77.919205  477.335815  109.428223   
15      2         2        car  440.188812  77.944565  477.276245  109.442551   
28      3         2        car  439.979523  78.024071  476.842804  109.332932   
39      4         2        car  439.934601  78.049248  476.764923  109.335083   
49      5         2        car  439.924896  78.103310  476.600525  109.249512   

      center_x   center_y    bottom_x    bottom_y lane_name direction  \
1   458.785706  93.673714  458.785706  109.428223    lane_1      down   
15  458.732544  93.693558  458.732544  109.442551    lane_1      down   
28  458.411163  93.678497  458.411163  109.332932    lane_1      down   
39  458.349762  93.692169  458.349762  109.335083    lane_1      down   
49  458.262695  93.676407  458.262695  109.249512    lane_1      down   

    wrong_lane  wrong_directio

,track_id,class_name,violation_type,first_frame,last_frame,violation_frames,lanes_used,direction
0,2,car,wrong_direction,1,526,312,"lane_1,lane_2",down
3,92,car,wrong_direction,91,188,19,unknown,up
6,671,motorcycle,wrong_direction,296,316,21,lane_2,down
11,928,motorcycle,wrong_direction,330,343,13,lane_2,down
20,1389,motorcycle,wrong_direction,546,615,59,lane_2,down
21,1403,motorcycle,wrong_direction,609,640,32,lane_2,down
24,1459,motorcycle,wrong_direction,570,596,17,lane_2,down
29,1573,motorcycle,wrong_lane,587,600,14,lane_2,up
31,1716,car,wrong_lane,620,632,13,lane_1,static_or_unclear
34,1764,motorcycle,wrong_lane,653,672,11,lane_2,up


# TRÍCH HÌNH ẢNH VI PHẠM

In [31]:
import cv2
import pandas as pd
from pathlib import Path

video_path = "20221003-115037.mp4"
violation_csv = "traffic_violations_detail.csv"

output_dir = Path("results/violation_frames")
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(violation_csv)

# Chỉ lấy frame có vi phạm
df = df[df["has_violation"] == True].copy()

# Đảm bảo kiểu dữ liệu
df["frame"] = df["frame"].astype(int)
df["track_id"] = df["track_id"].astype(int)

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError("Không mở được video")

for frame_id, g in df.groupby("frame"):
    # Nếu frame trong CSV bắt đầu từ 1 thì trừ 1 khi seek video
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id - 1)

    ret, frame = cap.read()
    if not ret:
        print(f"Không đọc được frame {frame_id}")
        continue

    for _, row in g.iterrows():
        x1 = int(row["x1"])
        y1 = int(row["y1"])
        x2 = int(row["x2"])
        y2 = int(row["y2"])

        track_id = int(row["track_id"])
        class_name = row["class_name"]
        violation_type = row["violation_type"]
        lane_name = row.get("lane_name", "unknown")

        label = f"ID {track_id} | {class_name} | {violation_type} | {lane_name}"

        # bbox đỏ cho phương tiện vi phạm
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)

        cv2.putText(
            frame,
            label,
            (x1, max(25, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 0, 255),
            2
        )

    output_path = output_dir / f"frame_{frame_id:06d}.jpg"
    cv2.imwrite(str(output_path), frame)

cap.release()

print("Saved violation frames to:", output_dir)

Saved violation frames to: results/violation_frames


# LƯU CLIP VI PHẠM

In [ ]:
import cv2
import pandas as pd
from pathlib import Path

video_path = "20221003-115037.mp4"

detail_csv = "traffic_violations_detail.csv"
summary_csv = "traffic_violations_summary.csv"

output_dir = Path("results/violation_clips")
output_dir.mkdir(parents=True, exist_ok=True)

df_detail = pd.read_csv(detail_csv)
df_summary = pd.read_csv(summary_csv)

df_detail["frame"] = df_detail["frame"].astype(int)
df_detail["track_id"] = df_detail["track_id"].astype(int)
df_summary["track_id"] = df_summary["track_id"].astype(int)

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError("Không mở được video")

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

for _, srow in df_summary.iterrows():
    track_id = int(srow["track_id"])
    class_name = srow["class_name"]
    violation_type = srow["violation_type"]

    start_frame = int(srow["first_frame"])
    end_frame = int(srow["last_frame"])

    g = df_detail[
        (df_detail["track_id"] == track_id) &
        (df_detail["frame"] >= start_frame) &
        (df_detail["frame"] <= end_frame)
    ].copy()

    if g.empty:
        continue

    bbox_by_frame = {
        int(row["frame"]): row
        for _, row in g.iterrows()
    }

    output_path = output_dir / f"id_{track_id}_{class_name}_{violation_type}.mp4"

    writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h)
    )

    for frame_id in range(start_frame, end_frame + 1):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id - 1)
        ret, frame = cap.read()

        if not ret:
            continue

        if frame_id in bbox_by_frame:
            row = bbox_by_frame[frame_id]

            x1 = int(row["x1"])
            y1 = int(row["y1"])
            x2 = int(row["x2"])
            y2 = int(row["y2"])

            label = f"ID {track_id} | {class_name} | {violation_type}"

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)

            cv2.putText(
                frame,
                label,
                (x1, max(15, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.4,
                (0, 0, 255),
                1,
                cv2.LINE_AA
            )

        writer.write(frame)


    writer.release()
    print("Saved:", output_path)

cap.release()

print("Total clips saved:", len(list(output_dir.glob("*.mp4"))))

Saved: results/violation_clips/id_2_car_wrong_direction.mp4
Saved: results/violation_clips/id_92_car_wrong_direction.mp4
Saved: results/violation_clips/id_671_motorcycle_wrong_direction.mp4
Saved: results/violation_clips/id_928_motorcycle_wrong_direction.mp4
Saved: results/violation_clips/id_1389_motorcycle_wrong_direction.mp4
Saved: results/violation_clips/id_1403_motorcycle_wrong_direction.mp4
Saved: results/violation_clips/id_1459_motorcycle_wrong_direction.mp4
Saved: results/violation_clips/id_1573_motorcycle_wrong_lane.mp4
Saved: results/violation_clips/id_1716_car_wrong_lane.mp4
Saved: results/violation_clips/id_1764_motorcycle_wrong_lane.mp4
Saved: results/violation_clips/id_1923_motorcycle_wrong_direction.mp4
Saved: results/violation_clips/id_2128_car_wrong_lane.mp4
Total clips saved: 12


: 